# LSTM Energy Forecasting Notebook

This notebook extends the baseline forecasting project with a TensorFlow LSTM model for time-series demand prediction.

It is designed as a next-step deep learning version of the project for students who want to explore sequence modeling in energy analytics.

In [ ]:
%pip install tensorflow pandas numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

from src.data_generator import generate_synthetic_energy_data

df = generate_synthetic_energy_data()
df = df[["timestamp", "energy_kwh"]].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()

In [ ]:
sequence_length = 24
values = df["energy_kwh"].values.astype(float)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values.reshape(-1, 1))

X, y = [], []
for i in range(len(scaled) - sequence_length):
    X.append(scaled[i:i + sequence_length])
    y.append(scaled[i + sequence_length])

X = np.array(X)
y = np.array(y)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(X_train.shape, X_test.shape)

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(sequence_length, 1)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

In [ ]:
pred_scaled = model.predict(X_test)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test.reshape(-1, 1))

mae = mean_absolute_error(actual, pred)
rmse = mean_squared_error(actual, pred, squared=False)
r2 = r2_score(actual, pred)

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R2: {r2:.4f}')

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(actual[:200], label='Actual', color='royalblue')
plt.plot(pred[:200], label='Predicted', color='red', linestyle='--')
plt.title('LSTM Forecast vs Actual Energy Demand')
plt.xlabel('Samples')
plt.ylabel('Energy (kWh)')
plt.legend()
plt.tight_layout()
plt.show()

## Interpretation

The LSTM model learns sequential dependencies in the energy-demand data and can often capture short-term temporal patterns better than a simple table-based regressor. This is a natural next step for students interested in deep learning and energy forecasting.